<a href="https://colab.research.google.com/github/aisaiahmaguadog/cmsi5350-notes/blob/main/Notebooks/Notebook_4_2_Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 Decision Tree

## Learning Objectives

Students will be able to:

* Load dataset with both `numpy` and `pandas.DataFrame`
* Visualize dataset with `matplotlib` and `seaborn`
* Construct and train a decision tree classifier using `sklearn`
* Evaluate model performance using standard metrics in `sklearn.metrics`
* Apply cross-validation techniques for robust model assessment
* Identify and mitigate overfitting in decision tree models


## Dataset Preparation

[Iris plant dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-plants-dataset) is a classic machine learning dataset containing 150 samples of iris flowers with measurements of sepal length, sepal width, petal length, and petal width for three species: versicolor, setosa, and virginica.

![Iris Image](https://miro.medium.com/v2/resize:fit:720/format:webp/1*ZK9_HrpP_lhSzTq9xVJUQw.png)

In [ ]:
# Load the Iris dataset
from sklearn import datasets

iris = datasets.load_iris()
print(iris.DESCR)

In [ ]:
# Load data using numpy
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names
print(f"X shape: {X.shape}", f"y shape: {y.shape}")
print(f"Feature Names: {feature_names}")
print(f"Target Names: {target_names}")

### Pandas DataFrame

Alternatively, we can also load the dataset using [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) - one of the most important data structures in Python for data analysis:

- **DataFrame**: A 2-dimensional labeled data structure with columns of potentially different types (like a spreadsheet or SQL table)
- **Common Operations**: Loading data from files (CSV, Excel), filtering, grouping, merging, and statistical analysis

In [ ]:
# Load data using panda.DataFrame
import pandas as pd

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
# Optional: Add target names instead of numbers
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# slicing in dataframe
df_setosa = df[df['species'] == 'setosa']
df_setosa.head()

### ========== TODO: START ========== ###
# TODO: other method to slice a dataframe based on the same condition?



### ========== TODO: END ========== ###


## ✅ Visualize the dataset

What are the effective ways to visualize the Iris dataset?

![Python Plots](https://media.geeksforgeeks.org/wp-content/uploads/20260604100327340518/plot_types.webp)


### ✅ Plot histograms of each feature's distribution in the Iris dataset.



In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(2, 2, figsize=(8, 6))
fig.suptitle('Iris Dataset Features Distribution', fontsize=16)

for idx, feature in enumerate(iris.feature_names):
    row, col = idx // 2, idx % 2  # Convert index to row, col position
    ax=axes[row, col]
    ### ========== TODO: START ========== ###


    ### ========== TODO: END ========== ###
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

### ✅ Plot 2x3 grid scatter plots that compares all measurement pairs, colored by species

In [ ]:
from itertools import combinations

### ========== TODO: START ========== ###
# TODO: Generate the feature pair combinations
# hint: use itertools.combinations() and feature_names


### ========== TODO: END ========== ###

# Create 3x2 subplot grid
fig, axes = plt.subplots(3, 2, figsize=(10, 15))
fig.suptitle('Iris Dataset: All Measurement Pairs', fontsize=16)


for idx, (x_col, y_col) in enumerate(pairs):
    row, col = idx // 2, idx % 2  # Convert index to row, col position
    ax=axes[row, col]

    ### ========== TODO: START ========== ###
    # TODO: Create scatter plots




    ### ========== TODO: END ========== ###
    ax.set_title(f'{x_col} vs {y_col}')

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(1.15, 0.95))

plt.tight_layout()
plt.show()

### `seaborn` Library

`seaborn` is a Python statistical data visualization library built on top of matplotlib. It provides a high-level interface for creating attractive and informative statistical graphics with minimal code.

* **Simplified syntax:** Create complex plots with just a few lines of code
* **Data-aware:** Works directly with pandas DataFrames and handles categorical data intelligently
* **Beautiful default styling:** Professional-looking plots out of the box
* **Color palettes:** Sophisticated [color palettes](https://seaborn.pydata.org/tutorial/color_palettes.html) that enhance readability


In [ ]:
import seaborn as sns
# Predefined palettes include: deep, muted, pastel, bright, dark, and colorblind
sns.set_palette("pastel")

# Load built-in dataset
df_sns = sns.load_dataset('iris')

# Create a scatter plot with species coloring - just one line!
sns.scatterplot(data=df_sns, x='sepal_length', y='petal_length', hue='species')
plt.show()

In [ ]:
# Same histogram with a different color palette
sns.set_palette("bright")

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
fig.suptitle('Iris Dataset Features Distribution', fontsize=16)

for idx, feature in enumerate(iris.feature_names):
    row, col = idx // 2, idx % 2  # Convert index to row, col position
    ax=axes[row, col]
    for species in iris.target_names:
        data = df[df['species'] == species][feature]
        ax.hist(data, alpha=0.7, label=species, bins=15)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Draw histogram with Seaborn

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
fig.suptitle('Iris Dataset Features Distribution', fontsize=16)

for idx, feature in enumerate(iris.feature_names):
    row, col = idx // 2, idx % 2  # Convert index to row, col position
    ax = axes[row, col]

    # Seaborn histplot with hue parameter
    sns.histplot(data=df, x=feature, hue='species', alpha=0.7, bins=15, ax=ax)

    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Basic Decision Tree Implementation

Let's create and train a basic decision tree classifier using the `sklearn` library.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"Feature count: {X_train.shape[1]}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Step 1: Create the decision tree
dt_classifier = DecisionTreeClassifier(random_state=42, criterion="entropy")

# Step 2: Train the mode using the training data
dt_classifier.fit(X_train, y_train)

# Step 3: Make predictions on the testing set
y_pred = dt_classifier.predict(X_test)

print(f"Predicted Y: {y_pred}")
print(f"True Y: {y_test}")

In [ ]:
from sklearn.metrics import accuracy_score

# Calculate accuracy
accuracy_test = accuracy_score(y_test, y_pred)
print(f"Testing accuracy: {accuracy_test:.4f}")

# Training performance
y_pred_train = dt_classifier.predict(X_train)
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training accuracy: {accuracy_train:.4f}") # What do you observe?

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n1. Confusion Matrix:")
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()



In [ ]:
# Precision, Recall, and F1 Score
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\n2. Overall Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Comprehensive classification report
print(f"\n3. Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

### Avoid overfitting - Pre-pruning (early stopping)

#### Key Parameters for Controlling Overfitting

- `max_depth` (default=None)
    - **Purpose**: Limits tree depth
    - **Example**: `max_depth=5`

- `min_samples_split` (default=2)
    - **Purpose**: Minimum samples required to split a node
    - **Example**: `min_samples_split=20`

- `min_samples_leaf` (default=1)
    - **Purpose**: Minimum samples required in each leaf
    - **Example**: `min_samples_leaf=10`

- `criterion` (default='gini')
    - **Options**: `'gini'`, `'entropy'`
    - **Purpose**: How to measure split quality

- `max_features` (default=None)
    - **Options**: `None`, `'sqrt'`, `'log2'`, int, float
    - **Purpose**: Number of features to consider per split

```python
from sklearn.tree import DecisionTreeClassifier

# Example
clf = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt'
)

# Small dataset
clf = DecisionTreeClassifier(max_depth=3, min_samples_split=10)

# Large dataset  
clf = DecisionTreeClassifier(max_depth=10, min_samples_split=50)
```



### ✅ Try to modify the hyperparameters and compare model performance

In [ ]:
# Create and train the decision tree

dt_classifier = DecisionTreeClassifier(random_state=42,
                                       # TODO: Add hyperparameters to prevent overfitting
                                       criterion=#TODO,
                                       max_depth=#TODO,
                                       min_samples_split= #TODO,
                                       min_samples_leaf= #TODO,
                                       max_features=#TODO
                                       )
# Train the mode using the training data
dt_classifier.fit(X_train, y_train)

# Make predictions on the testing set
y_pred = dt_classifier.predict(X_test)

# Calculate accuracy
accuracy_test = accuracy_score(y_test, y_pred)
print(f"Testing accuracy: {accuracy_test:.4f}")

# Training performance
y_pred_train = dt_classifier.predict(X_train)
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training accuracy: {accuracy_train:.4f}") # What do you observe?

### (Optional) Avoid overfitting - Post-pruning (reducing nodes)



In [ ]:
from sklearn.datasets import load_digits
# Load digits dataset (better for demonstrating pruning)
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, test_size=0.3, random_state=42)


# 1. Train full tree (no pruning)
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

# 2. Get pruning alphas
# More info about ccp_alphas: https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html
path = dt_full.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas
print(alphas)

# 3. Find best alpha by testing different values
best_score = 0
best_alpha = 0

for alpha in alphas:
    dt = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    dt.fit(X_train, y_train)
    score = dt.score(X_test, y_test)

    if score > best_score:
        best_score = score
        best_alpha = alpha

In [ ]:
# 4. Create final pruned tree
dt_pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
dt_pruned.fit(X_train, y_train)


# 5. Compare results
print("BEFORE PRUNING:")
print(f"Depth: {dt_full.get_depth()}")
print(f"Leaves: {dt_full.get_n_leaves()}")
print(f"Test accuracy: {dt_full.score(X_test, y_test):.3f}")

print("\nAFTER PRUNING:")
print(f"Depth: {dt_pruned.get_depth()}")
print(f"Leaves: {dt_pruned.get_n_leaves()}")
print(f"Test accuracy: {dt_pruned.score(X_test, y_test):.3f}")
print(f"Best alpha: {best_alpha:.6f}")

## Visualize the decision tree

In [ ]:
from sklearn.tree import plot_tree

# Visualize the decision tree
plt.figure(figsize=(15, 10))
plot_tree(dt_classifier,
          feature_names=feature_names,
          class_names=target_names,
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Decision Tree Visualization', fontsize=16)
plt.show()

In [ ]:
from sklearn.tree import export_text

# Export decision rules
tree_rules = export_text(dt_classifier, feature_names=feature_names)
print("Decision Tree Rules:")
print(tree_rules)

## Comprehensive model evaluation

In [ ]:
from sklearn.model_selection import cross_val_score

# Perform cross-validation
cv_scores = cross_val_score(dt_classifier, X_train, y_train, cv=10, scoring='accuracy')

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Visualize CV scores
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
plt.plot(range(1, 11), cv_scores, 'bo-')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(cv_scores, bins=5, alpha=0.7, color='skyblue', edgecolor='black')
plt.axvline(x=cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {cv_scores.mean():.4f}')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.title('Distribution of CV Scores')
plt.legend()

plt.tight_layout()
plt.show()